# Iteration  Analysis




In [9]:
from pathlib import Path
import json
import statistics
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

REVIEW_OUTPUTS_DIRS = [
    Path("./review_outputs_iter2/qwen2_5_7b"),
    Path("./review_outputs_iter2/qwen3_5_9b"),
    Path("./review_outputs_iter2/qwen3_8b"),
    Path("./review_outputs_iter2/llama3_1_8b"),
    Path("./review_outputs_iter2/phi4_reasoning_14b"),
    Path("./review_outputs_iter2/apertus"),
    Path("./review_outputs_iter2/qwen3_14b"),
]

ASSEMBLED_PAPERS_BASE_DIR = None 

In [10]:
def load_review_files(directories: list[Path]) -> list[dict]:
    data = []

    for directory in directories:
        files = sorted(directory.glob("*.review_pipeline.json"))

        if not files:
            print(f"Warning: no *.review_pipeline.json files found in {directory}")
            continue

        for path in files:
            with path.open("r", encoding="utf-8") as f:
                paper_data = json.load(f)

            paper_data["source_review_dir"] = str(directory)
            paper_data["generated_paper_model"] = directory.name
            paper_data["source_file"] = str(path)

            data.append(paper_data)

        print(f"Loaded {len(files)} paper review files from {directory}")

    if not data:
        raise FileNotFoundError(
            "No *.review_pipeline.json files found in any configured directory"
        )

    print(f"Loaded {len(data)} paper review files in total")
    return data


papers_raw = load_review_files(REVIEW_OUTPUTS_DIRS)

Loaded 40 paper review files from review_outputs_iter2/qwen2_5_7b
Loaded 40 paper review files from review_outputs_iter2/qwen3_5_9b
Loaded 40 paper review files from review_outputs_iter2/qwen3_8b
Loaded 40 paper review files from review_outputs_iter2/llama3_1_8b
Loaded 40 paper review files from review_outputs_iter2/phi4_reasoning_14b
Loaded 40 paper review files from review_outputs_iter2/apertus
Loaded 40 paper review files from review_outputs_iter2/qwen3_14b
Loaded 280 paper review files in total


In [11]:
def resolve_paper_path(paper: dict) -> Path | None:
    input_path = paper.get("input_path")

    if not input_path:
        return None

    p = Path(input_path)

    if p.exists():
        return p

    parts = list(p.parts)

    if parts and parts[0] == "assembled_papers_random":
        parts[0] = "assembled_papers"
        renamed_path = Path(*parts)

        if renamed_path.exists():
            return renamed_path

    return None

In [12]:
rows = []

for paper in papers_raw:
    meta = paper.get("meta_review", {})

    generator_model = paper.get("generated_paper_model")
    topic = None

    resolved_path = resolve_paper_path(paper)

    if resolved_path is not None:
        with resolved_path.open("r", encoding="utf-8") as f:
            original = json.load(f)

        generator_model = original.get("model") or generator_model
        topic = original.get("topic")

    rows.append({
        "paper_id": paper["paper_id"],
        "decision": meta.get("decision"),
        "generator_model": generator_model,
        "topic": topic.strip().lower() if topic else None,
    })

papers_df = pd.DataFrame(rows)

papers_df["accept_binary"] = (
    papers_df["decision"] == "accept"
).astype(int)

print(
    f"Papers: {len(papers_df)} | "
    f"topic resolved: {papers_df['topic'].notna().sum()} | "
    f"generator resolved: {papers_df['generator_model'].notna().sum()}"
)

Papers: 280 | topic resolved: 280 | generator resolved: 280


## Topic-Narrowing 


In [13]:
n_topics_total = papers_df["topic"].nunique()
accepted_df = papers_df[papers_df["decision"] == "accept"]
n_accepted_total = len(accepted_df)
n_topics_accepted = accepted_df["topic"].nunique()

print(f"Unique topics across ALL {len(papers_df)} papers: {n_topics_total}")
print(f"Unique topics among the {n_accepted_total} accepted: {n_topics_accepted}")
print(f"Narrowing ratio: {n_topics_accepted / n_topics_total:.1%}")

Unique topics across ALL 280 papers: 40
Unique topics among the 38 accepted: 26
Narrowing ratio: 65.0%


## Expected vs actual topic elimination


In [14]:
# Check 1: how many distinct generators wrote each topic's 7 papers?
topic_stats = papers_df.dropna(subset=["topic"]).groupby("topic").agg(
    n_papers=("paper_id", "count"),
    n_accepted=("accept_binary", "sum") if "accept_binary" in papers_df.columns
        else ("decision", lambda s: (s == "accept").sum()),
)
topic_stats["accept_rate"] = topic_stats["n_accepted"] / topic_stats["n_papers"]
topic_stats["eliminated"] = topic_stats["n_accepted"] == 0

generators_per_topic = papers_df.dropna(subset=["topic", "generator_model"]).groupby("topic")["generator_model"].nunique()

print("Distinct generators per topic (out of 7 papers):")
print(generators_per_topic.value_counts().sort_index())
print()
print(f"Mean distinct generators per topic: {generators_per_topic.mean():.2f} / 7")

Distinct generators per topic (out of 7 papers):
generator_model
3     4
4     9
5    23
6     4
Name: count, dtype: int64

Mean distinct generators per topic: 4.67 / 7


In [15]:
# Check 2: null model using each generator's own overall accept rate.
generator_accept_rate = papers_df.groupby("generator_model")["accept_binary"].mean()
print("Overall accept rate by generator:")
print(generator_accept_rate)
print()

def expected_elimination_prob(generators_in_topic: list[str]) -> float:
    p_all_reject = 1.0
    for g in generators_in_topic:
        p_all_reject *= (1 - generator_accept_rate.get(g, 0))
    return p_all_reject

topic_generators = papers_df.dropna(subset=["topic", "generator_model"]).groupby("topic")["generator_model"].apply(list)
expected_elim_prob_per_topic = topic_generators.apply(expected_elimination_prob)

expected_elimination_rate = expected_elim_prob_per_topic.mean()
actual_elimination_rate = topic_stats["eliminated"].mean()

print(f"Actual elimination rate:                    {actual_elimination_rate:.1%}")
print(f"Expected elimination rate under null model:  {expected_elimination_rate:.1%}")
print(f"(null model = chance alone, given each paper's own generator's overall accept rate)")

Overall accept rate by generator:
generator_model
hf:swiss-ai/Apertus-8B-Instruct-2509    0.025
llama3.1:8b                             0.050
phi4-reasoning:14b                      0.125
qwen2.5:7b                              0.075
qwen3.5:9b                              0.125
qwen3:14b                               0.275
qwen3:8b                                0.275
Name: accept_binary, dtype: float64

Actual elimination rate:                    35.0%
Expected elimination rate under null model:  36.1%
(null model = chance alone, given each paper's own generator's overall accept rate)
